# Unlimited-OCR — Plan B (transformers, no server) — FREE GPU

Runs `baidu/Unlimited-OCR` directly with `transformers.model.infer()` — **no SGLang, no tunnel**. It OCRs your documents on the GPU, saves one `.md` per doc, and zips them for download. You then fold them into the harness locally.

**Where to run (free):**
- **Colab (free):** Runtime -> Change runtime type -> **GPU (T4)**.
- **Kaggle (free, more VRAM):** New Notebook -> Settings -> Accelerator -> **GPU T4 x2**. Add your inputs as a *Dataset* and set `INPUTS` below to `/kaggle/input/<your-dataset>`.

**Do NOT use a TPU** — the model is CUDA-only.

T4 (15 GB) is enough for the single-page docs. The 10-page PDF (T6) may OOM on a single T4 — if so it's skipped/failed gracefully; use Kaggle's 2x T4 or a bigger GPU for it.

In [ ]:
# 1) GPU check (must show a GPU, not TPU)
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
# 2) Install transformers + the model's remote-code dependencies
!pip install -q transformers==4.57.1 accelerate addict einops easydict pymupdf==1.27.2.2 pillow matplotlib psutil
print('install done')

In [ ]:
# 3) Get your documents into an inputs/ folder.
#    COLAB: zip your local ai-ocr-field-test/inputs folder -> inputs.zip and upload it here.
#    KAGGLE: skip this cell, add a Dataset instead and set INPUTS below.
import os, glob, zipfile
try:
    from google.colab import files
    print('Upload inputs.zip (a zip of your inputs/ folder):')
    up = files.upload()
    for name in up:
        if name.endswith('.zip'):
            with zipfile.ZipFile(name) as z: z.extractall('.')
    # If the zip extracted files at top level instead of into inputs/, gather them.
    if not os.path.isdir('inputs'):
        os.makedirs('inputs', exist_ok=True)
        for ext in ('png','jpg','jpeg','pdf'):
            for f in glob.glob(f'*.{ext}'): os.rename(f, os.path.join('inputs', os.path.basename(f)))
except ImportError:
    print('Not on Colab — set INPUTS in the next cell to your Kaggle dataset path.')
print('inputs:', sorted(glob.glob('inputs/*')))

In [ ]:
# 4) Load the model (bf16). First run downloads ~6 GB.
import torch
from transformers import AutoModel, AutoTokenizer
MODEL = 'baidu/Unlimited-OCR'
tokenizer = AutoTokenizer.from_pretrained(MODEL, trust_remote_code=True)
model = AutoModel.from_pretrained(
    MODEL, trust_remote_code=True, torch_dtype=torch.bfloat16, low_cpu_mem_usage=True
).eval().cuda()
print('loaded ~%.1fB params' % (sum(p.numel() for p in model.parameters())/1e9))

In [ ]:
# 5) OCR every document -> unlimited_out/{test_id}_output.md  (+ a runlog sidecar)
import os, re, glob, time, csv, gc, io, contextlib, torch, fitz

INPUTS = 'inputs'            # Kaggle: change to '/kaggle/input/<your-dataset>'
OUT = 'unlimited_out'; os.makedirs(OUT, exist_ok=True)

def pdf_to_images(path, dpi=144, max_pages=10):
    out = []
    with fitz.open(path) as doc:
        for i, page in enumerate(doc):
            if i >= max_pages: break
            p = f'/tmp/{os.path.basename(path)}.{i}.png'
            page.get_pixmap(dpi=dpi).save(p); out.append(p)
    return out

def tid_of(f):
    m = re.match(r'(T\d+)', os.path.basename(f), re.I)
    return m.group(1).upper() if m else os.path.splitext(os.path.basename(f))[0]

files = [f for f in sorted(glob.glob(f'{INPUTS}/*')) if f.lower().endswith(('.png','.jpg','.jpeg','.pdf'))]
print(len(files), 'documents')
rows = []
for f in files:
    tid = tid_of(f); t = time.time(); status = 'success'; err = ''; text = ''
    try:
        # IMPORTANT: model.infer() PRINTS the result and returns None, so capture
        # stdout and prefer it; infer_multi() returns the text. Use whichever has content.
        buf = io.StringIO()
        with contextlib.redirect_stdout(buf):
            if f.lower().endswith('.pdf'):
                res = model.infer_multi(tokenizer, prompt='<image>Multi page parsing.',
                    image_files=pdf_to_images(f), output_path=OUT, image_size=1024,
                    max_length=8192, no_repeat_ngram_size=35, ngram_window=1024, save_results=False)
            else:
                res = model.infer(tokenizer, prompt='<image>document parsing.',
                    image_file=f, output_path=OUT, base_size=1024, image_size=640, crop_mode=True,
                    max_length=8192, no_repeat_ngram_size=35, ngram_window=128, save_results=False)
        captured = buf.getvalue().strip()
        ret = res if isinstance(res, str) else (res.get('text', '') if isinstance(res, dict) else '')
        text = ret.strip() if ret and ret.strip() else captured
    except Exception as e:
        status = 'failed'; err = str(e).replace(chr(10), ' ')[:300]; print('ERR', tid, err)
    open(f'{OUT}/{tid}_output.md', 'w', encoding='utf-8').write(text or '')
    lat = round(time.time() - t, 3)
    if not (text or '').strip() and status == 'success':
        status = 'failed'; err = 'empty output'
    rows.append({'timestamp': time.strftime('%Y-%m-%dT%H:%M:%S'), 'test_id': tid,
                 'input_file': os.path.basename(f), 'latency_sec': lat, 'status': status, 'error': err})
    print(f'{tid}: {status} {lat}s  ({len(text or "")} chars)')
    gc.collect(); torch.cuda.empty_cache()

with open(f'{OUT}/_unlimited_runlog.csv', 'w', newline='', encoding='utf-8') as fh:
    w = csv.DictWriter(fh, fieldnames=['timestamp','test_id','input_file','latency_sec','status','error'])
    w.writeheader(); w.writerows(rows)
print('done ->', OUT)

In [ ]:
# 6) Zip the outputs and download (Colab). Kaggle: find unlimited_ocr_outputs.zip in /kaggle/working.
import shutil
shutil.make_archive('unlimited_ocr_outputs', 'zip', 'unlimited_out')
try:
    from google.colab import files; files.download('unlimited_ocr_outputs.zip')
except Exception as e:
    print('Download unlimited_ocr_outputs.zip manually from the file browser.', e)

## Then, on your local machine
1. Unzip `unlimited_ocr_outputs.zip` into `ai-ocr-field-test/outputs/unlimited_ocr/`
   (so you have `outputs/unlimited_ocr/T1_output.md`, ... and `_unlimited_runlog.csv`).
2. Fold it into the report:
   ```
   venv\Scripts\python scripts\import_unlimited_ocr.py
   venv\Scripts\python scripts\init_metrics.py
   venv\Scripts\python scripts\generate_report.py
   venv\Scripts\python scripts\build_viewer.py
   ```
Unlimited-OCR now appears as the 4th model in `results/final_report.md` and the viewer.